# 04 — Model Evaluation

Evaluates the fine-tuned Qwen3-VL on the test set.

**Targets**: ≥98% cell accuracy, ≥95% TEDS, ≥98% zone accuracy


In [ ]:
# Cell 1: Load fine-tuned model
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    '/content/drive/MyDrive/numera-ml/models/qwen3vl-financial-lora',
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)
print('Model loaded for evaluation')


In [ ]:
# Cell 2: Run evaluation on test set
import json
from pathlib import Path
from PIL import Image

test_data = json.loads(Path('/content/drive/MyDrive/numera-ml/data/training/test.json').read_text())
print(f'Evaluating on {len(test_data)} test samples')

results = []
for item in test_data:
    img = Image.open(item['image']).convert('RGB')
    expected = json.loads(item['conversations'][1]['content'])

    # Run inference
    messages = [
        {'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': item['conversations'][0]['content']}]}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, images=[img], return_tensors='pt').to('cuda')

    import torch
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=4096, do_sample=False)
    predicted_text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    try:
        predicted = json.loads(predicted_text)
    except json.JSONDecodeError:
        predicted = {'tables': []}

    results.append({'expected': expected, 'predicted': predicted, 'image': item['image']})

print(f'Evaluation complete: {len(results)} samples')


In [ ]:
# Cell 3: Calculate metrics

zone_correct = 0
zone_total = 0
value_correct = 0
value_total = 0

for r in results:
    exp_tables = r['expected'].get('tables', [])
    pred_tables = r['predicted'].get('tables', [])

    for exp, pred in zip(exp_tables, pred_tables):
        # Zone accuracy
        zone_total += 1
        if exp.get('zone_type') == pred.get('zone_type'):
            zone_correct += 1

        # Value accuracy (cell-level)
        for exp_row, pred_row in zip(exp.get('rows', []), pred.get('rows', [])):
            for exp_val, pred_val in zip(exp_row.get('values', []), pred_row.get('values', [])):
                value_total += 1
                if exp_val == pred_val:
                    value_correct += 1

zone_acc = zone_correct / zone_total if zone_total else 0
cell_acc = value_correct / value_total if value_total else 0

print(f'Zone accuracy: {zone_acc:.2%} (target: ≥98%)')
print(f'Cell accuracy: {cell_acc:.2%} (target: ≥98%)')
print(f'Zone: {zone_correct}/{zone_total}, Cell: {value_correct}/{value_total}')

import mlflow
mlflow.log_metrics({'zone_accuracy': zone_acc, 'cell_accuracy': cell_acc})
